# Lab 1: LangChain Pipeline for BIT Professor Graphs

**Editable diagram:** [lab_1_workflow.excalidraw](lab_1_workflow.excalidraw)  
**Workflow preview:** [lab_1_workflow.svg](lab_1_workflow.svg)  
**Theory deck:** [lab1_theory_presentation.pptx](lab1_theory_presentation.pptx)

This lab walks one graph-first path: crawl professor pages, turn poster images into page notes, merge them into graph input, load Neo4j, then finish with a short MCP-backed agent pass.

You should leave with a practical sense of where plain Python, direct model calls, LCEL, and agents fit in one end-to-end workflow.


## 1. Environment Check

Goal: verify that the notebook is running in the expected project environment before any live calls.

Do: confirm paths, imports, and the active Python executable.

Expect: a healthy environment check or a clear warning that the wrong kernel is active.


In [1]:
import importlib.util
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / ".env").exists() else cwd.parent
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "lab1"
PROFESSOR_DIR = PROJECT_ROOT / "professors"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PROFESSOR_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook Python: {sys.executable}")

required_modules = [
    "langchain",
    "langchain_core",
    "langchain_openai",
    "langchain_mcp_adapters",
    "requests",
    "bs4",
    "dotenv",
    "kg_gen",
    "neo4j",
]

missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Missing modules: {missing}")

if ".venv" not in sys.executable:
    print("Warning: switch to the 'agents-tutorial (.venv)' kernel before continuing.")
else:
    print("Environment check passed.")

Notebook Python: /Users/khajievroma/Projects/agents_tutorial/.venv/bin/python
Environment check passed.


## 2. Services and Configuration

Goal: confirm the external systems this lab depends on.

Do: load environment settings, inspect the model and OCR configuration, and verify Neo4j connectivity.

Expect: a visible config summary plus a successful database connection test.


In [2]:
import subprocess

from neo4j import GraphDatabase

from bit_professor_chat.config import TutorSettings

settings = TutorSettings.from_env(PROJECT_ROOT / ".env")
ocr_config = settings.require_ocr()
settings.require_neo4j()

compose_status = subprocess.run(
    ["docker", "compose", "ps", "neo4j"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
if compose_status.returncode != 0:
    message = compose_status.stderr.strip() or compose_status.stdout.strip()
    raise RuntimeError(f"Neo4j Docker check failed: {message}")

compose_output = compose_status.stdout.lower()
if "agents-tutorial-neo4j" not in compose_output and "neo4j:latest" not in compose_output:
    raise RuntimeError(
        "Neo4j is not running. Start it with `docker compose up -d neo4j` and rerun this cell."
    )

driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_username, settings.neo4j_password),
)
try:
    driver.verify_connectivity()
finally:
    driver.close()

print(
    {
        "llm_base_url": settings.llm_base_url,
        "llm_model": settings.llm_model,
        "ocr_base_url": ocr_config["base_url"],
        "ocr_model": ocr_config["model"],
        "neo4j_uri": settings.neo4j_uri,
        "neo4j_database": settings.neo4j_database,
    }
)
print(compose_status.stdout)


{'llm_base_url': 'https://api.xiaocaseai.com/v1/', 'llm_model': 'qwen3.6-plus', 'ocr_base_url': 'https://api.xiaocaseai.com/v1/', 'ocr_model': 'qwen-vl-ocr-latest', 'neo4j_uri': 'bolt://localhost:7687', 'neo4j_database': 'neo4j'}
NAME                    IMAGE          COMMAND                  SERVICE   CREATED      STATUS                  PORTS
agents-tutorial-neo4j   neo4j:latest   "tini -g -- /startup…"   neo4j     5 days ago   Up 43 hours (healthy)   0.0.0.0:7474->7474/tcp, [::]:7474->7474/tcp, 0.0.0.0:7687->7687/tcp, [::]:7687->7687/tcp



## 3. Create the Base LangChain Model

Goal: create one shared LangChain model setup for the rest of the notebook.

Do: configure the reusable `ChatOpenAI` model and shared output parser.

Expect: later cells can focus on workflow logic instead of model boilerplate.


In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Reuse one shared chat model across the notebook so later cells focus on the workflow.
model = ChatOpenAI(
    model=settings.llm_model,
    base_url=settings.llm_base_url,
    api_key=settings.llm_api_key,
    temperature=0,
    timeout=120,
)
# Parse the model output into a plain string for the tiny LCEL example below.
text_parser = StrOutputParser()

print({"model_name": model.model_name, "base_url": settings.llm_base_url})


/Users/khajievroma/Projects/agents_tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model_name': 'qwen3.6-plus', 'base_url': 'https://api.xiaocaseai.com/v1/'}


## 4. Build a Tiny LCEL Chain

![Architecture step 04](workflow_steps/step_04.svg)

Goal: see LCEL in its smallest useful form before it appears inside the larger professor pipeline.

Do: compose `prompt | model | parser` into one runnable chain and invoke it on a short question.

Why this matters: LCEL is the cleanest way to express text-to-text stages. The later batch pipeline is more complex, but the same left-to-right composition idea is doing the work.

Expect: a short answer from one reusable runnable.


In [4]:
prompt = ChatPromptTemplate.from_template(
    """Answer in one short sentence.
    Question: {question}"""
)
# LCEL composes left to right here: prompt -> model -> parsed string.
explain_chain = prompt | model | text_parser

explanation = explain_chain.invoke(
    {
        "question": "Why does structured markdown help before knowledge graph generation?"
    }
).strip()
print(explanation)


It provides clear hierarchical organization and consistent formatting that reduces ambiguity and streamlines accurate entity and relationship extraction.


## 5. Crawl the Professor Listing Page

Goal: turn the BIT listing page into a reproducible set of professor profile links.

Do: fetch the directory page and extract professor names plus detail URLs.

Expect: an ordered list we can slice into a stable teaching sample.


In [5]:
from bit_professor_chat.source_discovery import (
    LISTING_URL,
    build_requests_session,
    collect_professor_links,
    discover_listing_pages,
)

DEFAULT_PROFESSOR_COUNT = 5

session = build_requests_session("agents-tutorial-lab1/0.1")
listing_pages = discover_listing_pages(LISTING_URL, session)
professor_links = collect_professor_links(listing_pages, session)
default_subset = professor_links[:DEFAULT_PROFESSOR_COUNT]

print(
    {
        "listing_page_count": len(listing_pages),
        "professor_count": len(professor_links),
        "default_subset_count": len(default_subset),
    }
)
for index, listing in enumerate(default_subset, start=1):
    print(f"{index}. {listing.name} -> {listing.detail_url}")

default_subset


{'listing_page_count': 4, 'professor_count': 44, 'default_subset_count': 5}
1. CHENG Cheng -> https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b113065.htm
2. CHE Haiying -> https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121632.htm
3. FILIPPO Fabrocini -> https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121633.htm
4. GAO Guangyu -> https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121634.htm
5. GAO Yang -> https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b113064.htm


[ProfessorListing(name='CHENG Cheng', detail_url='https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b113065.htm'),
 ProfessorListing(name='CHE Haiying', detail_url='https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121632.htm'),
 ProfessorListing(name='FILIPPO Fabrocini', detail_url='https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121633.htm'),
 ProfessorListing(name='GAO Guangyu', detail_url='https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121634.htm'),
 ProfessorListing(name='GAO Yang', detail_url='https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b113064.htm')]

## 6. Default 5-Professor Working Set and Worked Example

![Architecture step 06](workflow_steps/step_06.svg)

Goal: define the default 5-professor working set while keeping one professor as the worked example.

Do: select the first professor in the sample and extract that profile's image URLs.

Expect: one target professor plus the ordered poster images used in the next steps.


In [6]:
from bit_professor_chat.source_discovery import extract_image_urls

if not default_subset:
    raise ValueError("Could not find any professor links in the default working set.")

target_listing = default_subset[0]
image_urls = extract_image_urls(target_listing.detail_url, session)

print(
    {
        "worked_example": target_listing.name,
        "detail_url": target_listing.detail_url,
        "page_count": len(image_urls),
    }
)
image_urls


{'worked_example': 'CHENG Cheng', 'detail_url': 'https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b113065.htm', 'page_count': 1}


['https://isc.bit.edu.cn/images/2022-04/20180306024332179986.jpg']

## 7. Page-by-Page OCR With LangChain

Goal: convert each poster slice into inspectable page notes.

Do: call the multimodal model once per image, with stable OCR rules and page-specific input.

Why this matters: page-by-page OCR keeps the intermediate artifact readable and debuggable. It also makes the later merge step deterministic enough to explain live.

Expect: one markdown-style note block per page image.


In [7]:
import base64
import mimetypes
from typing import Any

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from openai import BadRequestError

from bit_professor_chat.ocr_transcript import (
    cleanup_markdown_artifact,
    normalize_page_ocr_markdown,
    page_markdown_to_page_extraction,
    render_page_notes,
)

OCR_SYSTEM_PROMPT = """You are an OCR transcription engine.

Transcribe only what is visible in the provided image.
Do not summarize, infer, correct, or complete missing text.
Return plain markdown only.
"""

PAGE_SLICE_OCR_PROMPT = """This is one image slice from a professor CV poster.

Preserve all visible text in reading order.
Use markdown headings for visible block titles and bullet lines for block content.
Never use tables, columns, code fences, LaTeX, prose, summaries, or explanations.
Do not infer missing text.
Keep cut-off lines exactly as visible.

Use this structure:
### <visible heading text or "(top identity block)" or "(continuation with no visible heading)">
- <transcribed line>
- <transcribed line>
"""

ocr_model = ChatOpenAI(
    model=ocr_config["model"],
    base_url=ocr_config["base_url"],
    api_key=ocr_config["api_key"],
    temperature=0,
    timeout=180,
)


def ocr_response_to_text(response: Any) -> str:
    """Normalize multimodal response content into plain markdown text."""
    content = response.content
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        chunks: list[str] = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                chunks.append(item.get("text", ""))
            else:
                chunks.append(str(item))
        return "\n".join(chunk for chunk in chunks if chunk).strip()
    return str(content).strip()


def image_url_to_data_url(image_url: str) -> str:
    response = session.get(image_url, timeout=60)
    response.raise_for_status()

    content_type = (response.headers.get("Content-Type") or "").split(";", maxsplit=1)[0].strip()
    if not content_type.startswith("image/"):
        guessed_type, _ = mimetypes.guess_type(image_url)
        content_type = guessed_type or "image/png"

    encoded = base64.b64encode(response.content).decode("ascii")
    return f"data:{content_type};base64,{encoded}"


def invoke_page_ocr(image_url: str) -> str:
    """Run OCR for one page image and clean the returned markdown."""

    # Keep stable OCR rules in the system role and page-specific input in the human role.
    messages = [
        SystemMessage(content=OCR_SYSTEM_PROMPT),
        HumanMessage(
            content=[
                {"type": "text", "text": PAGE_SLICE_OCR_PROMPT},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]
        ),
    ]

    try:
        response = ocr_model.invoke(messages)
    except BadRequestError as exc:
        error_text = str(exc).lower()
        if (
            "failed to download multimodal content" not in error_text
            and "invalidparameter" not in error_text
            and "invalid_parameter_error" not in error_text
        ):
            raise

        # Some providers fail to fetch remote image URLs, so retry with an inlined data URL.
        fallback_image_url = image_url_to_data_url(image_url)
        fallback_messages = [
            SystemMessage(content=OCR_SYSTEM_PROMPT),
            HumanMessage(
                content=[
                    {"type": "text", "text": PAGE_SLICE_OCR_PROMPT},
                    {"type": "image_url", "image_url": {"url": fallback_image_url}},
                ]
            ),
        ]
        response = ocr_model.invoke(fallback_messages)

    return cleanup_markdown_artifact(ocr_response_to_text(response))


def run_page_ocr(payload: dict[str, Any]) -> dict[str, Any]:
    raw_page_markdown = invoke_page_ocr(payload["image_url"])
    return {**payload, "raw_page_markdown": raw_page_markdown}


def normalize_page(payload: dict[str, Any]) -> dict[str, Any]:
    normalized_page_markdown = normalize_page_ocr_markdown(
        raw_text=payload["raw_page_markdown"],
        page_number=payload["page_number"],
    )
    return {**payload, "normalized_page_markdown": normalized_page_markdown}


def parse_page(payload: dict[str, Any]) -> dict[str, Any]:
    page_extraction = page_markdown_to_page_extraction(
        normalized_markdown=payload["normalized_page_markdown"],
        page_number=payload["page_number"],
        image_url=payload["image_url"],
    )
    return {**payload, "page_extraction": page_extraction}


ocr_page_chain = (
    RunnableLambda(run_page_ocr)
    | RunnableLambda(normalize_page)
    | RunnableLambda(parse_page)
)

page_payloads = [
    {"page_number": index, "image_url": image_url}
    for index, image_url in enumerate(image_urls, start=1)
]
page_results = [ocr_page_chain.invoke(payload) for payload in page_payloads]
page_extractions = [result["page_extraction"] for result in page_results]
page_notes_preview = render_page_notes(page_extractions)

print(
    {
        "ocr_model": ocr_model.model_name,
        "page_count": len(page_extractions),
        "message_roles": ["system", "human"],
        "first_page_headings": [block.heading_text for block in page_extractions[0].blocks[:5]],
    }
)
print(page_notes_preview[:2500])


{'ocr_model': 'qwen-vl-ocr-latest', 'page_count': 1, 'message_roles': ['system', 'human'], 'first_page_headings': ['(top identity block)']}
<!-- PAGE 1 START -->
# Page Extraction
## Source
- page_number: 1
- image_url: https://isc.bit.edu.cn/images/2022-04/20180306024332179986.jpg
## Visible Blocks
### Block 1
- heading_text: (top identity block)
- block_role: standalone
- content:
  - 姓名(Name): 程成
  - 职称(title): 副教授
  - Associate Professor
  - Cheng Cheng
  - 学院(School): 计算机
  - 专业(I): 计算机科学与技术
  - Computer Science & Technology
  - 研究方向(Research
  - 联系方式()
  - 电话(): 13011008553
  - Interests
  - 邮箱
  - Human-computer
  - (E-mail): guoguocheng@vip.sina.com,
  - Interaction
  - cc@bit.edu.cn
  - 2.虚拟环境
  - Virtual Environments
  - 3.虚拟制造
  - 照片(photo)
  - Virtual Manufacturing
  - 个人简历(Biography)
  - 教育背景:
  - 1999/9-2003/2: Human Computer Interaction, Ph.D. degree of Computer Science at the Institute of Software, Chinese Academy of Sciences, Beijing, China;
  - 1996/9-1999/7: Computer

## 8. Helper Merge Into Graph Input

![Architecture step 08](workflow_steps/step_08.svg)

Goal: hand off from page notes to graph input without adding another durable artifact.

Do: save the page notes for debugging, recover the top block if needed, and merge the notes into one transient graph-ready text payload.

Expect: a page-notes artifact plus one merged text preview for the worked example.


In [8]:
import re
from typing import Any

from langchain_core.runnables import RunnableLambda

from bit_professor_chat.ingestion import build_graph_input_text_from_page_notes
from bit_professor_chat.ingestion_models import ProfessorPageNotesResult
from bit_professor_chat.ocr_transcript import (
    cleanup_markdown_artifact,
    extract_header_identity_lines,
    needs_top_block_fallback,
    render_page_notes,
)


def recover_top_block(payload: dict[str, Any]) -> dict[str, Any]:
    # Fallback OCR pass for the compact top identity block when page notes miss it.
    header_lines = extract_header_identity_lines(
        model=ocr_model,
        image_url=payload["image_url"],
        session=session,
    )
    return {**payload, "header_lines": header_lines}


header_recovery_chain = RunnableLambda(recover_top_block)
page_notes_markdown = cleanup_markdown_artifact(render_page_notes(page_extractions))
supplemental_header_lines: list[str] = []

# Only run the extra OCR call when the worked example is missing the expected header info.
if needs_top_block_fallback(pages=page_extractions, expected_name=target_listing.name):
    supplemental_header_lines = header_recovery_chain.invoke({"image_url": image_urls[0]})["header_lines"]

worked_slug = re.sub(r"[^a-z0-9]+", "-", target_listing.name.lower()).strip("-")
worked_page_notes_path = ARTIFACT_DIR / f"{worked_slug}-page-notes.md"
worked_page_notes_path.write_text(page_notes_markdown, encoding="utf-8")

worked_page_notes_result = ProfessorPageNotesResult(
    name=target_listing.name,
    detail_url=target_listing.detail_url,
    slug=worked_slug,
    page_count=len(image_urls),
    image_urls=list(image_urls),
    page_notes_path=str(worked_page_notes_path.relative_to(PROJECT_ROOT)),
    page_notes_markdown=page_notes_markdown,
    supplemental_header_lines=supplemental_header_lines,
)

# Merge the page notes and any recovered header lines into graph-ready text.
graph_input_text = build_graph_input_text_from_page_notes(
    page_notes_result=worked_page_notes_result,
)

print(
    {
        "worked_example_page_notes": worked_page_notes_result.page_notes_path,
        "page_count": worked_page_notes_result.page_count,
        "graph_input_char_count": len(graph_input_text),
        "used_header_fallback": bool(supplemental_header_lines),
    }
)
print(graph_input_text[:2500])


08:46:09 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: _ssl.c:999: The handshake operation timed out. Falling back to local backup.


{'worked_example_page_notes': 'artifacts/lab1/cheng-cheng-page-notes.md', 'page_count': 1, 'graph_input_char_count': 2993, 'used_header_fallback': True}
# Cheng Cheng

## Basic Information
- 姓氏(Chinese): 程成
- 英文名: Cheng Cheng
- 照片 (photo)
- 教育背景
- 1999/9-2003/2: Human Computer Interaction, Ph.D. degree of Computer Science at the Institute of Software, Chinese Academy of Sciences, Beijing, China;
- 1996/9-1999/7: Computer graphics and CAD, Master degree of Computer Science at the Department of Computer Science, Shandong University, Jinan, China;
- 1984/9-1988/7: Control Theory, Bachelor degree at the Department of Computer Science, Nankai University, Tianjin, China;
- 工作经历
- 2003/3-now: Associate Professor in the Department of Computer Science and Technology, Beijing Institute of Technology;
- 1994/7-1996/7: Computer system engineer in Hainan zhongshang Futures Exchange;
- 1992/7-1994/7: Computer engineer and system developer in TianLong photoelectric company;
- 1988/7-1992/7: Mechanica

## 9. Parallel KG-Gen Pipelines

Goal: scale the worked-example flow to the default professor batch.

Do: run one per-professor pipeline across the first five professors with bounded concurrency and one retry path.

Expect: a batch summary with success or error status and graph artifact locations.


In [ ]:
from langchain_core.runnables import RunnableLambda

from bit_professor_chat.graph_ingestion import (
    generate_professor_graph,
    insert_professor_graph,
    require_kg_gen,
    reset_neo4j_database,
    save_graph_json,
)
from bit_professor_chat.ingestion import (
    build_graph_input_text_from_page_notes,
    prepare_professor_page_notes,
)
from bit_professor_chat.ingestion_models import ProfessorGraphIngestionResult
from bit_professor_chat.legacy_cache import build_cached_summary_line
from bit_professor_chat.markdown_corpus import slugify_name
from bit_professor_chat.source_discovery import build_requests_session

GRAPH_NAME = "BIT_CSAT_LAB1"
MAX_CONCURRENCY = 4
RESET_NEO4J_BEFORE_BATCH = True

artifact_dir = ARTIFACT_DIR
artifact_dir.mkdir(parents=True, exist_ok=True)


def prepare_professor_for_graph(listing):
    # Fetch one professor and produce page notes with the OCR pipeline.
    worker_session = build_requests_session("agents-tutorial-lab1-worker/0.1")
    page_notes_result = prepare_professor_page_notes(
        listing=listing,
        ocr_model=ocr_model,
        session=worker_session,
        project_root=PROJECT_ROOT,
        artifact_namespace="lab1",
    )
    return {"listing": listing, "page_notes_result": page_notes_result}


def build_graph_input(payload):
    # Convert page notes into the single text blob used for graph generation.
    graph_input_text = build_graph_input_text_from_page_notes(
        page_notes_result=payload["page_notes_result"],
    )
    return {**payload, "graph_input_text": graph_input_text}


def generate_graph(payload):
    # Run KG generation for one professor and save the graph artifacts.
    page_notes_result = payload["page_notes_result"]
    graph_input_text = payload["graph_input_text"]
    graph_json_path = artifact_dir / f"{page_notes_result.slug}-graph.json"
    graph_html_path = artifact_dir / f"{page_notes_result.slug}-graph.html"
    graph = generate_professor_graph(graph_input_text, settings)
    save_graph_json(graph, graph_json_path)
    require_kg_gen().visualize(graph, str(graph_html_path), open_in_browser=False)
    return {
        **payload,
        "graph": graph,
        "graph_json_path": str(graph_json_path.relative_to(PROJECT_ROOT)),
        "graph_html_path": str(graph_html_path.relative_to(PROJECT_ROOT)),
    }


def insert_graph(payload):
    # Load the generated graph into Neo4j and assemble the result record.
    listing = payload["listing"]
    page_notes_result = payload["page_notes_result"]
    graph_input_text = payload["graph_input_text"]
    graph = payload["graph"]
    insert_professor_graph(
        graph=graph,
        settings=settings,
        graph_name=GRAPH_NAME,
        professor_name=listing.name,
        detail_url=listing.detail_url,
    )
    summary_line = build_cached_summary_line(graph_input_text, listing.name)
    return ProfessorGraphIngestionResult(
        name=listing.name,
        detail_url=listing.detail_url,
        slug=page_notes_result.slug,
        page_count=page_notes_result.page_count,
        page_notes_path=page_notes_result.page_notes_path,
        graph_json_path=payload["graph_json_path"],
        graph_html_path=payload["graph_html_path"],
        entity_count=len(graph.entities),
        relation_count=len(graph.relations),
        summary_line=summary_line,
        graph_input_char_count=len(graph_input_text),
        validation_status="valid",
        validation_notes=[],
    )


# Each runnable matches one stage in the lab flow: page notes -> graph input -> KG -> Neo4j.
professor_pipeline = (
    RunnableLambda(prepare_professor_for_graph)
    | RunnableLambda(build_graph_input)
    | RunnableLambda(generate_graph)
    | RunnableLambda(insert_graph)
)


def run_professor_with_retry(listing, max_attempts: int = 2):
    # Retry the whole professor pipeline once so failures stay isolated per professor.
    slug = slugify_name(listing.name)
    graph_json_path = artifact_dir / f"{slug}-graph.json"
    graph_html_path = artifact_dir / f"{slug}-graph.html"

    for attempt in range(max_attempts):
        try:
            result = professor_pipeline.invoke(listing)
            if attempt == 0:
                return result
            return ProfessorGraphIngestionResult(
                **{
                    **result.to_dict(),
                    "retry_count": attempt,
                }
            )
        except Exception as exc:
            if attempt + 1 < max_attempts:
                continue
            return ProfessorGraphIngestionResult(
                name=listing.name,
                detail_url=listing.detail_url,
                slug=slug,
                page_count=0,
                page_notes_path=str((artifact_dir / f"{slug}-page-notes.md").relative_to(PROJECT_ROOT)),
                graph_json_path=str(graph_json_path.relative_to(PROJECT_ROOT)),
                graph_html_path=str(graph_html_path.relative_to(PROJECT_ROOT)),
                entity_count=0,
                relation_count=0,
                summary_line=f"{listing.name} could not be summarized because ingestion failed.",
                retry_count=max_attempts - 1,
                graph_input_char_count=0,
                status="error",
                error=str(exc),
                validation_status="error",
                validation_notes=[str(exc)],
            )


if RESET_NEO4J_BEFORE_BATCH:
    reset_neo4j_database(settings)

batch_results = RunnableLambda(run_professor_with_retry).batch(
    list(default_subset),
    config={"max_concurrency": MAX_CONCURRENCY},
)

target_graph_result = next(result for result in batch_results if result.name == target_listing.name)
batch_summary = [
    {
        "name": result.name,
        "status": result.status,
        "retry_count": result.retry_count,
        "page_count": result.page_count,
        "entity_count": result.entity_count,
        "relation_count": result.relation_count,
        "page_notes_path": result.page_notes_path,
        "graph_json_path": result.graph_json_path,
        "error": result.error,
    }
    for result in batch_results
]

print(
    {
        "graph_name": GRAPH_NAME,
        "max_concurrency": MAX_CONCURRENCY,
        "success_count": sum(1 for result in batch_results if result.status == "ok"),
        "error_count": sum(1 for result in batch_results if result.status != "ok"),
        "retrying_professors": sum(1 for result in batch_results if result.retry_count > 0),
        "worked_example_graph": target_graph_result.graph_json_path,
    }
)
batch_summary


## 10. Verify Neo4j After Parallel Load

![Architecture step 10](workflow_steps/step_10.svg)

Goal: verify the Neo4j load before switching to question answering.

Do: query node counts, relationship counts, loaded professors, and a few sample facts.

Expect: direct evidence that the batch pipeline wrote structured graph data.


In [ ]:
from neo4j import GraphDatabase

from bit_professor_chat.graph_ingestion import verify_neo4j_graph

# First run the coarse verification helper over the loaded graph.
verification = verify_neo4j_graph(settings)

driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_username, settings.neo4j_password),
)
# Then run a direct Cypher sample so students can inspect real triples in Neo4j.
with driver.session(database=settings.neo4j_database) as neo4j_session:
    sample_facts = [
        record.data()
        for record in neo4j_session.run(
            """
            MATCH (s:Entity)-[r]->(o:Entity)
            WHERE r.graph_name = $graph_name
            RETURN s.name AS subject, r.predicate AS predicate, o.name AS object
            LIMIT 8
            """,
            graph_name=GRAPH_NAME,
        )
    ]
driver.close()

print(verification)
sample_facts


## 11. Short MCP Epilogue

Goal: show the payoff after ingestion by asking the graph a real question through MCP tools.

Do: construct the MCP client, expose Neo4j tools, create the notebook-local agent, and run one query.

Why this matters: this is an epilogue, not a second lab. The point is to show how the graph becomes usable once the data pipeline is already in place.

Expect: a graph-backed answer plus a visible tool trace summary.


In [ ]:
import json
import time
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware.tool_retry import ToolRetryMiddleware
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

GRAPH_AGENT_SYSTEM_PROMPT = """
You are the BIT Professor Graph Assistant for Beijing Institute of Technology.

You answer student questions by querying Neo4j through the MCP tools provided to you.
Use only database evidence returned by the tools. If the graph does not contain an answer,
say that clearly instead of guessing.

Current graph conventions:
- Most nodes use the label `Entity`.
- The primary text field is `name`.
- Root professor nodes carry `professor_name` and `detail_url`.
- Nodes and relationships may carry `source_professors`, which list the professor dossiers that contributed the fact.
- Relationship types are normalized uppercase versions of the original predicate.
- The original natural-language predicate is also stored on each relationship as `predicate`.

Working style:
1. Inspect the schema first when the shape of the graph is uncertain.
2. Resolve the professor or topic with simple Cypher first, then drill down.
3. When multiple independent read-only graph checks would help, prefer parallel tool calls in one turn instead of serializing them.
4. Prefer simple `MATCH ... RETURN ... LIMIT ...` queries over clever Cypher.
5. If a Cypher query fails, read the error, simplify the query, and retry.
6. Never use `RETURN` inside a list comprehension; Cypher list comprehensions use `|`.
7. Prefer concise answers with concrete names and relationships.
8. Mention when an answer is partial because the graph is incomplete.
""".strip()

mcp_connection = settings.neo4j_mcp_connection()

# Mask the password before printing the MCP command that the notebook will use.
display_args = []
hide_next = False
for arg in mcp_connection["args"]:
    if hide_next:
        display_args.append("***")
        hide_next = False
        continue
    display_args.append(arg)
    if arg == "--password":
        hide_next = True

print({"command": mcp_connection["command"], "args": display_args})

question = "What are three example subject-predicate-object facts from the current BIT_CSAT_LAB1 graph?"
agent_model = ChatOpenAI(
    model=settings.llm_model,
    base_url=settings.llm_base_url,
    api_key=settings.llm_api_key,
    temperature=0,
)
# Build the same MCP-backed agent inline so the notebook can inspect its loop directly.
mcp_client = MultiServerMCPClient({"neo4j": mcp_connection}, tool_name_prefix=True)
mcp_tools = await mcp_client.get_tools()
graph_tool_names = [tool.name for tool in mcp_tools]
graph_agent = create_agent(
    agent_model,
    mcp_tools,
    system_prompt=GRAPH_AGENT_SYSTEM_PROMPT,
    middleware=[ToolRetryMiddleware(max_retries=1, on_failure="continue")],
    name="bit_professor_mcp_agent",
)


def truncate_text(text: str, limit: int = 1800) -> str:
    if len(text) <= limit:
        return text
    return text[: limit - 3] + "..."


def stringify_agent_content(content: Any) -> str:
    """Normalize mixed message content into printable text."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                text = item.get("text") or item.get("content")
                parts.append(text if text else json.dumps(item, ensure_ascii=False))
            else:
                parts.append(str(item))
        return "\n".join(part for part in parts if part)
    if isinstance(content, dict):
        return json.dumps(content, ensure_ascii=False, indent=2)
    return str(content)


def extract_agent_final_answer(messages) -> str:
    """Pick the final AI answer after tool use is finished."""
    for message in reversed(messages):
        if message.__class__.__name__ != "AIMessage":
            continue
        if getattr(message, "tool_calls", None):
            continue
        return stringify_agent_content(message.content)
    return stringify_agent_content(messages[-1].content) if messages else ""


def extract_agent_tool_traces(messages) -> list[dict[str, Any]]:
    """Match each ToolMessage back to the tool call that produced it."""
    tool_args_by_call_id: dict[str, dict[str, Any]] = {}
    traces: list[dict[str, Any]] = []

    for message in messages:
        for tool_call in getattr(message, "tool_calls", []) or []:
            tool_args_by_call_id[tool_call["id"]] = {
                "name": tool_call["name"],
                "args": tool_call.get("args", {}),
            }

        if message.__class__.__name__ != "ToolMessage":
            continue

        tool_call_id = getattr(message, "tool_call_id", "")
        tool_metadata = tool_args_by_call_id.get(tool_call_id, {})
        traces.append(
            {
                "name": getattr(message, "name", None) or tool_metadata.get("name", "tool"),
                "args": tool_metadata.get("args", {}),
                "content_preview": truncate_text(stringify_agent_content(message.content)),
            }
        )

    return traces


def extract_agent_llm_turn_count(messages) -> int:
    return sum(1 for message in messages if message.__class__.__name__ == "AIMessage")


def extract_agent_tool_calls_per_turn(messages) -> list[int]:
    """Count how many tool calls the model emitted in each tool-using turn."""
    return [
        len(message.tool_calls)
        for message in messages
        if message.__class__.__name__ == "AIMessage" and getattr(message, "tool_calls", None)
    ]


async def ask_graph_agent(question: str, conversation: list[dict[str, str]] | None = None) -> dict[str, Any]:
    """Run one graph question and return the answer plus loop-inspection metadata."""
    messages = list(conversation or [])
    messages.append({"role": "user", "content": question})
    start_time = time.perf_counter()
    result = await graph_agent.ainvoke({"messages": messages})
    elapsed_seconds = time.perf_counter() - start_time
    result_messages = result["messages"]
    tool_calls_per_turn = extract_agent_tool_calls_per_turn(result_messages)
    return {
        "answer": extract_agent_final_answer(result_messages),
        "tool_traces": extract_agent_tool_traces(result_messages),
        "elapsed_seconds": elapsed_seconds,
        "available_tools": list(graph_tool_names),
        "llm_turn_count": extract_agent_llm_turn_count(result_messages),
        "tool_call_count": sum(tool_calls_per_turn),
        "tool_calls_per_turn": tool_calls_per_turn,
    }


# Run one agent turn here; the next cell inspects this exact result in more detail.
agent_reply = await ask_graph_agent(question)

print(
    {
        "available_tools": graph_tool_names,
        "question": question,
        "tools_used": [trace["name"] for trace in agent_reply["tool_traces"]],
    }
)
print(agent_reply["answer"])


## 12. Inspect MCP Tools and Agent Loop

Goal: make the agent loop inspectable instead of magical.

Do: list the available MCP tools, time a run, and print the ordered tool trace before the final answer.

Why this matters: the core loop is `model -> tool call(s) -> tool result(s) -> model`, and this cell lets students see that sequence directly.

Expect: tool inventory, timing, turn and tool counts, and a final answer.


In [ ]:
if "agent_reply" not in globals():
    agent_reply = await ask_graph_agent(question)

print(
    {
        "available_tools": agent_reply["available_tools"],
        "elapsed_seconds": round(agent_reply["elapsed_seconds"], 3),
        "llm_turn_count": agent_reply["llm_turn_count"],
        "tool_call_count": agent_reply["tool_call_count"],
        "tool_calls_per_turn": agent_reply["tool_calls_per_turn"],
    }
)

if not agent_reply["tool_traces"]:
    print("The agent answered without calling MCP tools.")
else:
    for index, trace in enumerate(agent_reply["tool_traces"], start=1):
        print(f"\nStep {index}: {trace['name']}")
        print(json.dumps(trace["args"], ensure_ascii=False, indent=2))
        print(trace["content_preview"])

print("\nFinal answer:")
print(agent_reply["answer"])


## Appendix A. Optional Inline Gradio UI

Goal: package the same notebook-local agent in a lightweight UI without changing the core logic.

Do: reuse the existing agent and launch a small inline Gradio chat.

Expect: an optional demo surface with example prompts and latest-trace visibility.


In [ ]:
import gradio as gr

GRADIO_EXAMPLES = [
    "Which research areas are mentioned for CHE Haiying?",
    "What organizations or institutions is CHE Haiying connected to?",
    "Show a few triples related to CHE Haiying.",
]
TRACE_PLACEHOLDER = {
    "message": "Send a question to inspect the latest MCP tool trace.",
}


def history_to_conversation(history) -> list[dict[str, str]]:
    # Normalize Gradio chat history back into plain role/content messages.
    messages: list[dict[str, str]] = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in {"user", "assistant"} and content:
                messages.append({"role": role, "content": str(content)})
            continue
        if isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item
            if user_message:
                messages.append({"role": "user", "content": str(user_message)})
            if assistant_message:
                messages.append({"role": "assistant", "content": str(assistant_message)})
    return messages


async def ask_graph(question: str, history):
    prompt = question.strip()
    history = history_to_conversation(history)
    if not prompt:
        return "", history, TRACE_PLACEHOLDER

    try:
        reply = await ask_graph_agent(prompt, conversation=history)
    except Exception as exc:
        error_message = f"Query failed: {exc}"
        updated_history = history + [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": error_message},
        ]
        return "", updated_history, {"error": error_message}

    updated_history = history + [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": reply["answer"]},
    ]
    # Show the latest tool trace from the same notebook-local agent run.
    trace_view = reply["tool_traces"] or {"message": "The agent answered without calling MCP tools."}
    return "", updated_history, trace_view


with gr.Blocks() as demo:
    gr.Markdown(
        """### BIT Professor Graph Chat
Ask the notebook-local LangChain + MCP agent a question and inspect the latest Neo4j tool trace."""
    )
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=420,
                value=[],
            )
            prompt = gr.Textbox(
                label="Ask a graph question",
                placeholder="Try one of the example prompts below.",
            )
            with gr.Row():
                ask_button = gr.Button("Ask", variant="primary")
                reset_button = gr.Button("Start a new chat")
            gr.Examples(examples=GRADIO_EXAMPLES, inputs=prompt)
        with gr.Column(scale=2):
            trace_view = gr.JSON(
                label="Latest MCP tool trace",
                value=TRACE_PLACEHOLDER,
            )
            gr.Markdown(
                "Tool traces show the most recent MCP calls used to answer the question. "
                "They help connect the notebook workflow to the UI layer."
            )

    ask_button.click(
        ask_graph,
        inputs=[prompt, chatbot],
        outputs=[prompt, chatbot, trace_view],
    )
    prompt.submit(
        ask_graph,
        inputs=[prompt, chatbot],
        outputs=[prompt, chatbot, trace_view],
    )
    reset_button.click(
        lambda: ("", [], TRACE_PLACEHOLDER),
        outputs=[prompt, chatbot, trace_view],
    )

demo.launch(inline=True)


## 13. Wrap-Up

![Architecture step 13](workflow_steps/step_13.svg)

Goal: leave with a clean mental model of the full Lab 1 pipeline.

Do: connect the crawl, OCR, merge, graph load, and MCP agent stages back into one story.

Expect: a concise summary of where plain Python, model calls, LCEL, and agents each fit.
